# 04 — Anti-Spoofing (Liveness Detection)

Binary classifier: **real** vs **fake** (printed/screen-displayed face).

**Dataset**: Download CelebA-Spoof or NUAA from Kaggle and set `DATA_PATH` below.

Expected folder structure:
```
anti_spoof_data/
├── real/   # real face images
└── fake/   # spoofed face images
```

In [ ]:
import os
from pathlib import Path

# Use GPU card #2 (index 1). Change to '0' to use the first card.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "1")

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score
)
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, models, transforms
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# ⚠️  Set this to your spoof dataset folder
DATA_PATH   = Path("../data/anti_spoof_data")
WEIGHTS_DIR = Path("../weights")
WEIGHTS_DIR.mkdir(exist_ok=True)

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE   = 112
BATCH_SIZE = 64
EPOCHS     = 15
LR         = 1e-3
VAL_SPLIT  = 0.2

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

print(f"Device: {DEVICE}")
print(f"CUDA_VISIBLE_DEVICES: {os.environ.get('CUDA_VISIBLE_DEVICES')}")
print(f"Data  : {DATA_PATH}  (exists: {DATA_PATH.exists()})")

## Model Definition

In [ ]:
class AntiSpoofNet(nn.Module):
    """MobileNetV2-based binary (real/fake) classifier."""

    def __init__(self):
        super().__init__()
        base = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        # Replace classifier: 1280 -> 2
        base.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(1280, 2),
        )
        self.net = base

    def forward(self, x):
        return self.net(x)


## Data Loading

In [ ]:
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

full_ds = datasets.ImageFolder(DATA_PATH, transform=train_tf)
print(f"Classes : {full_ds.classes}")
print(f"Total   : {len(full_ds)} images")

n_val   = int(len(full_ds) * VAL_SPLIT)
n_train = len(full_ds) - n_val
train_ds, val_ds = random_split(full_ds, [n_train, n_val])
val_ds.dataset = datasets.ImageFolder(DATA_PATH, transform=val_tf)  # val uses no aug

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
print(f"Train: {n_train} | Val: {n_val}")


## Training

In [ ]:
model     = AntiSpoofNet().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss()

history = {"train_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in tqdm(train_loader, leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(imgs)
        correct    += (out.argmax(1) == labels).sum().item()
        total      += len(imgs)

    # Validation
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            out = model(imgs)
            val_correct += (out.argmax(1) == labels).sum().item()
            val_total   += len(imgs)

    tr_loss = total_loss / total
    tr_acc  = correct / total
    val_acc = val_correct / val_total
    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["val_acc"].append(val_acc)
    scheduler.step()
    print(f"Epoch {epoch:02d}/{EPOCHS} | loss {tr_loss:.4f} | train acc {tr_acc:.3f} | val acc {val_acc:.3f}")

torch.save(model.state_dict(), WEIGHTS_DIR / "anti_spoofing.pth")
print("Saved anti_spoofing.pth")


## Evaluation

In [ ]:
model.eval()
all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(DEVICE)
        out  = model(imgs)
        prob = torch.softmax(out, dim=1)[:, 1].cpu().numpy()  # P(real)
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(prob)

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

print(classification_report(all_labels, all_preds, target_names=["fake", "real"]))

# APCER, BPCER, ACER
# APCER = FPR among real (attacks that pass as real)
# BPCER = FNR among fake (real rejected as fake)
real_mask = all_labels == 1
fake_mask = all_labels == 0
APCER = (all_preds[fake_mask] == 1).mean()   # fake predicted as real
BPCER = (all_preds[real_mask] == 0).mean()   # real predicted as fake
ACER  = (APCER + BPCER) / 2
print(f"APCER={APCER:.4f}  BPCER={BPCER:.4f}  ACER={ACER:.4f}")

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=["fake", "real"], yticklabels=["fake", "real"])
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title("Anti-Spoof Confusion Matrix")
plt.tight_layout()
plt.show()
